# TFM — Recomendación personalizada frente a elección fija de métodos de aprendizaje supervisado

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cecimerelo/ml-sandbox/blob/main/notebooks/tfm_graficos.ipynb)

Este cuaderno reproduce los gráficos y el contraste estadístico del Trabajo Fin de Máster a
partir de los resultados del estudio benchmark, y sigue la estructura exigida. La sección 1
describe el origen de los datos, qué datasets se usaron y por qué forman una muestra amplia y
diversa. La sección 2 presenta la técnica: un benchmark de 20 métodos de ISLR evaluados con
validación cruzada (10 o 5 particiones) y la evaluación *leave-one-dataset-out* de cuatro
estrategias de selección de método. La sección 3 recoge los resultados y el contraste de las
hipótesis, y la sección 4 comprueba su robustez frente a valores faltantes y el coste de exigir
interpretabilidad.

Elegir qué método de aprendizaje supervisado usar para un problema concreto no es sencillo,
sobre todo para quien empieza. El trabajo aborda esa elección con dos piezas: un estudio
benchmark, que entrena y evalúa muchos métodos en muchos datasets para saber cuál funciona mejor
en cada uno, y una aplicación web en la que el usuario describe su problema mediante un
formulario, o sube sus datos, y recibe tres métodos recomendados con una explicación, que
después puede entrenar sobre sus propios datos. La recomendación se organiza en dos capas. La
primera, Layer 1, aplica las reglas de selección de método de ISLR (*An Introduction to
Statistical Learning*), el libro de referencia del curso, junto con las restricciones que
indique el usuario, como necesitar un modelo interpretable. La segunda, **Layer 2**, es el
objetivo del TFM: un modelo entrenado sobre los resultados del benchmark que, a partir de las
características del problema del usuario, predice qué métodos van a funcionar mejor. En los
gráficos aparece como *Aprendida (Layer 2)*, siempre en azul. Las demás estrategias son los
puntos de comparación contra los que se mide.

La hipótesis principal es que la recomendación personalizada, basada en las características del
problema, elige el mejor método con más frecuencia que una recomendación fija. La hipótesis
secundaria es que las heurísticas didácticas de ISLR aciertan más que el azar.

Todo el código del estudio está en
[github.com/cecimerelo/ml-sandbox](https://github.com/cecimerelo/ml-sandbox). Basta con ejecutar
las celdas en orden (*Entorno de ejecución → Ejecutar todas*). La aplicación, con el
formulario, está desplegada en [ml-sandbox-7cn3.onrender.com](https://ml-sandbox-7cn3.onrender.com).
Está en el plan gratuito de Render, que apaga el servicio cuando nadie lo usa, así que la
primera carga puede tardar alrededor de un minuto.

## Índice

- [0. Preparación](#preparacion)
   - [Las estrategias de selección de método](#estrategias)
- [1. Origen de los datos](#datos)
   - [Gráfico 1 · Variedad de la colección](#grafico-1)
   - [Gráfico 2 · Composición de la colección](#grafico-2)
- [2. Técnica: el benchmark](#tecnica)
   - [Gráfico 3 · Qué método gana más veces](#grafico-3)
   - [Gráfico 4 · Pérdida de cada método en cada dataset](#grafico-4)
   - [Cómo se mide cada estrategia](#metricas)
- [3. Resultados y contraste de las hipótesis](#resultados)
   - [Tabla de resultados principal](#tabla-resultados)
   - [Gráfico 5 · Aciertos con intervalos de confianza](#grafico-5)
   - [Contraste de la hipótesis principal](#contraste-principal)
   - [Contraste de la hipótesis secundaria](#contraste-secundaria)
   - [Gráfico 6 · Distribución del regret](#grafico-6)
   - [Gráfico 7 · Aprendida frente a fija, dataset a dataset](#grafico-7)
   - [Gráfico 8 · Resultados por tamaño del dataset](#grafico-8)
- [4. Robustez](#robustez)
   - [4.1 Valores faltantes · Gráfico 9](#grafico-9)
   - [4.2 Coste de exigir interpretabilidad · Gráfico 10](#interpretabilidad)
- [5. Conclusiones](#conclusiones)
- [Anexo 1 · Recalcular todo desde los resultados](#anexo)
- [Anexo 2 · Decisiones metodológicas y trabajo futuro](#anexo-2)

<a id="preparacion"></a>

## 0. Preparación

En Colab todas las dependencias vienen instaladas. En local pueden instalarse con Poetry desde
`notebooks/` (`poetry install`), eligiendo después `notebooks/.venv` como kernel, o puede usarse
cualquier entorno de Python 3.12 o posterior, ya que la primera celda instala con `pip` lo que
falte.

El cuaderno clona el repositorio, que es público, para disponer de los resultados. El benchmark
completo, con 38.820 evaluaciones, tardó horas, así que el cuaderno parte de sus resultados
guardados. El fichero `data/results/results-801f29e62585.parquet` contiene una fila por dataset,
método, porcentaje de faltantes y partición. A partir de él se calculan otros tres:
`outcomes.parquet`, con lo que eligió cada estrategia en cada dataset en la evaluación
*leave-one-dataset-out*. `method_scores.parquet` tiene la puntuación media de cada método en cada
dataset y si cuenta entre los mejores. `explainability.json` recoge el coste de exigir
interpretabilidad. Los tres se generan con `scripts/export_outcomes.py`, y la sección final
explica cómo recalcularlos.

Que estos tres ficheros se hayan calculado después del benchmark no afecta a los resultados,
porque no son una reconstrucción aproximada de lo ocurrido durante el estudio, sino el mismo
cálculo repetido. Son deterministas: se obtienen del parquet de resultados con el código del
repositorio y una semilla fija (20260815), de modo que con los mismos resultados dan siempre
exactamente las mismas cifras, se calculen el día del benchmark o meses después. Además, así se
calculó siempre. El estudio nunca guardó lo que eligió cada estrategia en su momento, sino que,
para cada dataset, las estrategias se reconstruyen a partir de los resultados de los otros 105 y
se observa qué eligen. Las cifras de los informes del estudio se obtuvieron con ese mismo
procedimiento, y estos ficheros solo guardan además el detalle por dataset en lugar de
únicamente la media. Por último, está comprobado: repetir el cálculo sobre la copia de los
resultados con la que se redactaron las conclusiones originales reproduce exactamente sus
cifras, por ejemplo el 0,47 de hit rate de la estrategia aprendida y de la fija.

Algunas cifras de este cuaderno difieren ligeramente de esas conclusiones originales por un
cambio en los datos, no en el cálculo. Ridge elige su parámetro de regularización (`alpha`)
probando varios valores por validación cruzada. Al principio solo probaba los tres que trae
scikit-learn por defecto (0,1, 1 y 10). Más tarde se amplió a 25 valores entre 0,001 y 1.000 y se
volvió a ejecutar solo Ridge, que con más opciones mejoró en 10 de los 106 datasets. Como acertar
significa elegir uno de los mejores métodos de cada dataset, esa mejora cambia en algunos
datasets qué métodos cuentan como mejores y mueve ligeramente los aciertos de las estrategias.
La aprendida, por ejemplo, pasa de 0,47 a 0,49. Las conclusiones no cambian, y este cuaderno usa
los resultados corregidos.

La siguiente celda comprueba que están instaladas las librerías que usa el cuaderno, instala con
`pip` las que falten y las importa todas.

In [1]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

# Instala lo que falte en el entorno actual, para poder usar el cuaderno sin Poetry.
# módulo que se importa → paquete que se instala
# nbformat y jinja2 no se usan directamente: Plotly necesita nbformat para mostrar los gráficos
# en el cuaderno, y pandas necesita jinja2 para dar formato a las tablas (`.style`).
DEPENDENCIAS = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow",
                "plotly": "plotly", "scipy": "scipy", "nbformat": "nbformat>=4.2.0",
                "jinja2": "jinja2>=3.1"}
faltan = [paquete for modulo, paquete in DEPENDENCIAS.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--disable-pip-version-check", *faltan], check=True)
    # Sin esto, Python puede no ver los paquetes recién instalados en esta misma sesión.
    importlib.invalidate_caches()

import nbformat  # noqa: F401  (lo requiere fig.show() de Plotly)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

Los datos del estudio viven en el repositorio. Si el cuaderno se ejecuta dentro de una copia
local se usa esa y, si no, como ocurre en Colab, se clona desde GitHub.

In [2]:
REPO = "https://github.com/cecimerelo/ml-sandbox.git"

def encontrar_repo() -> Path:
    # En local el cuaderno vive en notebooks/; en Colab se clona el repositorio.
    for candidato in (Path.cwd(), Path.cwd().parent):
        if (candidato / "config" / "collection.json").exists():
            return candidato
    destino = Path("/content/ml-sandbox")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(destino)], check=True)
    return destino

ROOT = encontrar_repo()
print("Repositorio en", ROOT)

Repositorio en /Users/cecimerelo/Desktop/personal/TFM


Ahora cargamos los resultados del benchmark, lo que eligió cada estrategia, las puntuaciones
por método y la descripción de la colección (datasets, meta-características y métodos). La salida
confirma el tamaño del estudio.

In [3]:
RESULTADOS = pd.read_parquet(ROOT / "data/results/results-801f29e62585.parquet")
OUTCOMES = pd.read_parquet(ROOT / "data/results/outcomes.parquet")
METODOS_SCORES = pd.read_parquet(ROOT / "data/results/method_scores.parquet")
EXPLICABILIDAD = json.loads((ROOT / "data/results/explainability.json").read_text())
COLECCION = json.loads((ROOT / "config/collection.json").read_text())
METAFEATURES = pd.DataFrame(json.loads((ROOT / "config/metafeatures.json").read_text())["datasets"])
METODOS = pd.DataFrame(json.loads((ROOT / "config/methods.json").read_text())["methods"])

print(f"{len(RESULTADOS):,} evaluaciones · {RESULTADOS.dataset.nunique()} datasets · "
      f"{RESULTADOS.method.nunique()} métodos · faltantes: "
      f"{sorted(RESULTADOS.missing_rate.unique())}")

38,820 evaluaciones · 106 datasets · 20 métodos · faltantes: [np.float64(0.0), np.float64(0.05), np.float64(0.25)]


<a id="estrategias"></a>

### Las estrategias de selección de método

Cada estrategia recibe las características de un dataset y devuelve una ordenación de los
métodos candidatos, del más al menos recomendado. Todas se construyen igual: para evaluar un
dataset solo pueden aprender de los otros 105 (*leave-one-dataset-out*), de modo que ninguna ha
visto el problema sobre el que se la juzga. En los datos aparecen con su nombre en el código,
que se indica entre paréntesis.

El método *leave-one-dataset-out* se ha usado porque reproduce la situación real del
recomendador: un usuario llega con su dataset, que el modelo nunca ha visto, y lo que se quiere
saber es si en ese caso acierta. Por eso, para cada uno de los 106 datasets, la estrategia se
entrena con los otros 105 y se le pide una recomendación para el que se ha dejado fuera. Se
separan datasets enteros y no filas al azar porque el modelo aprende de una tabla con una fila
por dataset y método. Con una separación al azar, parte de los métodos de un mismo dataset
caería en entrenamiento y parte en prueba, y el modelo recordaría ese dataset en lugar de
predecirlo, dando una cifra mejor de la real sin que nada fallara. El procedimiento se aplica a
todas las estrategias, también a las que no aprenden, para que ninguna parta con ventaja, y fue
precisamente este diseño el que destapó un error real: la estrategia fija elegía su método
mirando también el dataset con el que se la iba a evaluar, lo que la habría hecho
artificialmente difícil de batir, y se corrigió. Por último, se deja fuera un dataset cada vez y
no un grupo porque, con solo 106 datasets, así cada modelo se entrena con el máximo posible
(105) y es el más parecido al que usa la aplicación, entrenado con los 106. Además, el resultado
no depende de cómo se agrupen los datasets y el coste, 106 entrenamientos de un modelo pequeño,
es de pocos minutos.

La primera estrategia es el modelo desarrollado y entrenado en este TFM, y las otras cuatro
existen para poder juzgarlo: dos cotas de referencia (la fija y la aleatoria), la alternativa
que ofrece el libro de texto (las heurísticas) y la variante que usa la aplicación (la híbrida).

🎯 La estrategia **aprendida** (`learned (Layer 2)`) es el recomendador del TFM, la recomendación
personalizada y el objeto de estudio del trabajo. Es un *random forest* entrenado sobre los
resultados del benchmark que aprende a predecir cuánto rendimiento perderá cada método frente al
mejor a partir de las siete meta-características del dataset: el tipo de tarea, la banda de
filas, la banda de predictores, el régimen de dimensionalidad (observaciones por predictor), los
tipos de variable, el nivel de valores faltantes y el balance de clases. Con esa predicción
recomienda los métodos de menor pérdida prevista.

Se ha usado un *random forest* porque lo que el modelo tiene que aprender depende de
combinaciones de características: tener pocas filas no afecta igual a una red neuronal, que
necesita muchos datos, que a un árbol de decisión. Un modelo lineal solo suma efectos por
separado, así que habría que indicarle a mano cada una de esas combinaciones. Un bosque de
árboles las encuentra por sí mismo, ya que cada rama divide primero por una característica y
después por otra. El bosque es deliberadamente pequeño (300 árboles y al menos 3 observaciones
por hoja) porque la tabla de entrenamiento también lo es, unas 1.500 filas, una por cada
combinación de dataset y método. Con un bosque más profundo, el modelo memorizaría los datasets
concretos en lugar de aprender el patrón común a todos ellos.

El modelo predice cuánto pierde cada método frente al mejor, y no directamente cuál es el
ganador, por dos motivos. Con la regla de empate muchos datasets tienen varios ganadores, y un
clasificador aprendería a considerar incorrecto un método que en realidad empata con el mejor.
Además, la pérdida prevista ordena directamente las alternativas que muestra la aplicación.
Medirla frente al mejor método de cada dataset, en lugar de usar la puntuación bruta, evita que
el modelo dedique su capacidad a aprender qué datasets son fáciles, algo que no sirve para
recomendar.

La elección del random forest responde a este razonamiento, pero no se comparó con otros modelos
que también podrían servir para esta tarea, como *gradient boosting* o una regresión lineal con
interacciones. Es posible que alguno de ellos rinda mejor, y comprobarlo queda como línea de
trabajo futuro.

La estrategia del **método único fijo** (`single best method`) no mira el problema: recomienda
siempre el mismo orden de métodos, el de menor pérdida media frente al mejor en los datasets de
entrenamiento. Es la comparación clave de la hipótesis, porque si personalizar no la supera, no
merece la pena inspeccionar el problema del usuario. En este estudio ese método resulta ser
Random Forest en todos los casos.

Las **heurísticas de ISLR** (`heuristics (ISLR)`) son las reglas de decisión didácticas de *An
Introduction to Statistical Learning*, basadas en el equilibrio entre sesgo y varianza, los
supuestos de cada método o el tamaño de muestra frente al número de predictores, codificadas
como reglas fijas. Son la primera capa de la aplicación, y puntúan cada método según las
características del problema. Por ejemplo, con pocas observaciones favorecen los métodos
sencillos o regularizados, porque los flexibles acaban ajustando el ruido. Cuando el número de
predictores se acerca al de observaciones, favorecen la regularización, como Ridge o Lasso. Y
con muchas observaciones por predictor favorecen los métodos flexibles, que ya tienen datos
suficientes. No aprenden nada de los datos y representan lo que diría el libro.

La estrategia **aleatoria** (`random choice`) ordena los métodos al azar, con semilla fija, y es
la cota inferior: lo que se obtiene sin mirar nada.

La estrategia **híbrida** (`hybrid`) es la que usa la aplicación. Las restricciones del usuario,
como exigir un modelo interpretable, descartan métodos, y la estrategia aprendida ordena los que
quedan. Sin restricciones es idéntica a la aprendida, por lo que se omite de los gráficos y solo
interviene en el coste de la interpretabilidad del Gráfico 10.

La celda siguiente prepara lo que comparten todos los gráficos. Fija el color y la etiqueta en
español de cada estrategia, que son los mismos en todas las figuras, define la función `estilo`
que les da formato y construye la tabla `OUT`, con una fila por dataset y estrategia que indica
si acertó y cuánto perdió. `OUT0` es esa misma tabla sin valores faltantes, la que usan los
resultados principales. La estrategia híbrida se descarta aquí, después de comprobar que
coincide con la aprendida.

In [4]:
# Estilo común de los gráficos
FUENTE = "system-ui, -apple-system, 'Segoe UI', sans-serif"
TINTA, TINTA_2, TINTA_3 = "#0b0b0b", "#52514e", "#898781"
REJILLA, EJE, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
AZULES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

ESTRATEGIAS = {  # nombre en el código → (etiqueta, color)
    "learned (Layer 2)": ("Aprendida (Layer 2)", AZUL),
    "single best method": ("Método único fijo", NARANJA),
    "heuristics (ISLR)": ("Heurísticas (ISLR)", AQUA),
    "random choice": ("Aleatoria", TINTA_3),
}
ORDEN_ESTRATEGIAS = [etiqueta for etiqueta, _ in ESTRATEGIAS.values()]
COLOR_ESTRATEGIA = {etiqueta: color for etiqueta, color in ESTRATEGIAS.values()}
ETIQUETA_METODO = dict(zip(METODOS.name, METODOS.label, strict=True))
BANDAS_FILAS = ["<500", "500-10k", ">10k"]

hibrida = OUTCOMES[OUTCOMES.strategy == "hybrid"].set_index(["missing_rate", "dataset"]).choice
aprendida = (OUTCOMES[OUTCOMES.strategy == "learned (Layer 2)"]
             .set_index(["missing_rate", "dataset"]).choice)
# Sin restricciones, la híbrida elige exactamente lo mismo que la aprendida; por eso se omite
# de los gráficos. Si algún cambio en el código rompiera esa igualdad, esta línea lo avisaría.
assert (hibrida == aprendida).all(), "La híbrida ya no coincide con la aprendida: revisar"

OUT = OUTCOMES[OUTCOMES.strategy.isin(ESTRATEGIAS)].copy()
OUT["estrategia"] = OUT.strategy.map(lambda s: ESTRATEGIAS[s][0])
OUT = OUT.merge(METAFEATURES[["dataset", "rows", "task"]], on="dataset", how="left")
OUT0 = OUT[OUT.missing_rate == 0.0]


MARGEN_SUP = 150


def estilo(fig, titulo, subtitulo=None, alto=460):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:13px;color:{TINTA_2}'>{subtitulo}</span>"
    fig.update_layout(
        template="simple_white",
        font=dict(family=FUENTE, size=13, color=TINTA),
        title=dict(text=texto, x=0.01, xanchor="left", y=0.97, yanchor="top"),
        paper_bgcolor=FONDO, plot_bgcolor=FONDO, height=alto,
        margin=dict(t=MARGEN_SUP, l=70, r=30, b=60),
        # La leyenda va entre el subtítulo y el área de dibujo, por encima de los títulos
        # de los subgráficos: se desplaza en píxeles, convertidos a la escala del papel.
        legend=dict(orientation="h", x=0, y=1 + 42 / (alto - MARGEN_SUP - 60),
                    yanchor="bottom", title=None, font=dict(color=TINTA_2)),
        hoverlabel=dict(font=dict(family=FUENTE, size=13), bgcolor="white"),
    )
    fig.update_xaxes(linecolor=EJE, tickcolor=EJE, tickfont_color=TINTA_2,
                     title_font_color=TINTA_2, showgrid=False)
    fig.update_yaxes(linecolor=EJE, tickcolor=EJE, tickfont_color=TINTA_2,
                     title_font_color=TINTA_2, showgrid=True, gridcolor=REJILLA)
    fig.update_annotations(font=dict(color=TINTA_2))
    return fig

<a id="datos"></a>

## 1. Origen de los datos

La idea era conseguir datasets que cubrieran cada uno de los casos que el usuario podría llegar
a elegir a través del formulario.

La colección combina tres fuentes públicas. [OpenML-CC18](https://www.openml.org/s/99) es de
clasificación y [OpenML-CTR23](https://www.openml.org/s/353) de regresión, las dos con 500 filas
o más por dataset. [PMLB](https://github.com/EpistasisLab/pmlb) aporta datasets de ambas tareas
con menos de 500 filas.

Un dataset es elegible si cumple todos estos criterios:

- Tiene entre 50 y 100.000 filas. Por debajo de 50, cada partición de la validación cruzada de 5
  particiones tendría menos de 10 filas. Por encima de 100.000, un solo dataset consumiría más
  cómputo que toda la banda pequeña.
- Tiene como mucho 500 predictores, el mismo límite que acepta la aplicación.
- No está construido a partir de imágenes (MNIST, Fashion-MNIST, Devnagari-Script, CIFAR-10).
- Contiene datos medidos, no generados. Quedan fuera los sintéticos de PMLB: las ecuaciones de
  física de Feynman y Strogatz y los datasets BNG, muestreados de redes bayesianas.
- Sus propios autores no lo han retirado. PMLB marca como obsoletos algunos datasets.
- Tiene completos los metadatos, incluido el número de filas.
- Si viene de PMLB, tiene menos de 500 filas, porque esa colección solo cubre la banda que no
  cubren OpenML-CC18 ni OpenML-CTR23.

Además, de cada familia de datasets (varias versiones de la misma fuente, como los
`analcatdata_*` o los `mfeat-*`) entran como mucho dos.

Los criterios no tienen todos el mismo origen. El límite de 500 predictores y la exclusión de
imágenes son el alcance que el documento de requisitos de la aplicación ya declaraba antes del
estudio, así que la selección no puede haberse ajustado para favorecer a ningún método. El
umbral de 500 filas es la banda más baja del formulario y coincide con el criterio de los
autores de OpenML-CC18, que descartan como demasiado pequeño todo dataset por debajo de esa
cifra ([generador de la colección](https://github.com/openml/benchmark-suites/blob/master/OpenML%20Benchmark%20generator.ipynb)).
El resto son decisiones propias del estudio, justificadas en
[`DECISIONS.md`](https://github.com/cecimerelo/ml-sandbox/blob/main/DECISIONS.md) (D-004, D-016,
D-025 y D-054). La de las familias es la más débil: la familia se deduce del nombre del dataset
y el límite de dos no responde a ningún criterio externo.

De 267 candidatos, 195 cumplían los criterios de elegibilidad y se conservaron 106,
estratificados por tamaño y tipo de tarea con un objetivo de 20 datasets por combinación. Para
que el cómputo fuera asumible, cada dataset se evalúa como mucho sobre 20.000 filas, elegidas al
azar con la semilla del estudio. Como el límite es el mismo para todos los métodos, no cambia la
comparación entre ellos. La colección no se reunió de una vez. La primera versión del estudio
tenía 60 datasets, y las primeras evaluaciones se hicieron incluso antes de completarla, con 44.
Después se amplió a 106, ampliando por igual todos los grupos de tamaño y de tarea, para
comprobar si los resultados de esa primera versión se mantenían con una muestra casi el doble de
grande. Varios de ellos no se mantuvieron, como se explica en las secciones siguientes.

<a id="grafico-1"></a>

Para que la comparación generalice, los datasets deben cubrir tamaños y dimensiones muy
distintos, y el Gráfico 1 permite comprobar que la colección es variada. Cada punto es un
dataset, situado según su número de filas y de predictores, ambos en escala logarítmica. El
color indica la fuente y la forma, el tipo de tarea.

In [5]:
datasets = pd.DataFrame(COLECCION["datasets"])
FUENTES = {"openml-cc18": "OpenML-CC18", "openml-ctr23": "OpenML-CTR23", "pmlb": "PMLB"}
TAREAS = {"regression": "Regresión", "binary classification": "Clasificación binaria",
          "multiclass classification": "Clasificación multiclase"}
datasets["fuente"] = datasets.source.map(FUENTES)
datasets = datasets.merge(METAFEATURES[["dataset", "task", "rows"]].rename(
    columns={"dataset": "name", "task": "tarea_mf", "rows": "banda_filas"}), on="name")
datasets["tarea"] = datasets.tarea_mf.map(TAREAS)

fig = px.scatter(
    datasets, x="rows", y="predictors", color="fuente", symbol="tarea",
    log_x=True, log_y=True, hover_name="name",
    hover_data={"rows": ":,", "predictors": True, "fuente": True, "tarea": True},
    color_discrete_sequence=[AZUL, NARANJA, AQUA],
    symbol_sequence=["circle", "diamond", "square"],
    labels={"rows": "Filas (escala log)", "predictors": "Predictores (escala log)",
            "fuente": "Fuente", "tarea": "Tarea"},
)
fig.update_traces(marker=dict(size=10, opacity=0.85, line=dict(width=1.5, color=FONDO)))
estilo(fig, f"Gráfico 1 · Los {len(datasets)} datasets van de {datasets.rows.min():,} a "
            f"{datasets.rows.max():,} filas y de {datasets.predictors.min()} a "
            f"{datasets.predictors.max()} predictores".replace(",", "."),
       "Cada punto es un dataset. Color: fuente · forma: tipo de tarea", alto=540).show()

<a id="grafico-2"></a>

El Gráfico 2 y la tabla que lo acompaña muestran cuántos datasets hay de cada tamaño y tipo de
tarea, y de qué fuente sale cada uno. La colección se construyó estratificando por esas dos
variables, con un objetivo de 20 datasets por cada combinación de banda de filas y tipo de
tarea. En las dos bandas de menos de 10.000 filas el objetivo se alcanzó exactamente, con 40
datasets en cada una. En la banda de más de 10.000 filas no había suficientes candidatos
elegibles: se incluyeron todos y solo sumaron 26, 11 de clasificación y 15 de regresión.

El equilibrio se limita a esas dos variables. Hay regímenes con pocos representantes, porque
no se estratificó por ellos: solo 6 datasets tienen más de 50 predictores, 13 están en el
régimen de alta dimensionalidad (pocas observaciones por predictor, todos con menos de 500
filas) y solo 8 tienen valores faltantes de origen, algo que el estudio compensa en parte con
las versiones con un 5 % y un 25 % de celdas eliminadas. Las conclusiones descansan por tanto en
pocos casos en esos regímenes y deben leerse con más cautela en ellos.

In [6]:
recuento = (datasets.groupby(["banda_filas", "tarea"]).size().rename("n").reset_index())
fig = px.bar(
    recuento, x="banda_filas", y="n", color="tarea", text="n",
    category_orders={"banda_filas": BANDAS_FILAS, "tarea": list(TAREAS.values())},
    color_discrete_sequence=[AZUL, NARANJA, AQUA],
    labels={"banda_filas": "Filas del dataset", "n": "Datasets", "tarea": "Tarea"},
)
fig.update_traces(textposition="inside", textfont_color="white",
                  marker_line=dict(width=2, color=FONDO))
estilo(fig, "Gráfico 2 · Composición de la colección",
       f"Datasets por tamaño y tipo de tarea · total {len(datasets)}").show()

pd.crosstab(datasets.fuente, datasets.tarea, margins=True, margins_name="Total")

tarea,Clasificación binaria,Clasificación multiclase,Regresión,Total
fuente,,,,
OpenML-CC18,18,13,0,31
OpenML-CTR23,0,0,35,35
PMLB,10,10,20,40
Total,28,23,55,106


<a id="tecnica"></a>

## 2. Técnica: el benchmark

Cada uno de los 20 métodos del temario de ISLR se entrenó y evaluó con validación cruzada en cada
dataset aplicable, usando *balanced accuracy* en clasificación y R² en regresión (se justifican
más abajo). El número de
particiones depende de la fuente. En los 66 datasets de OpenML (CC18 y CTR23) se usan las 10
particiones predefinidas que publica OpenML, para que los resultados sean directamente
comparables con los trabajos publicados sobre esas colecciones. En los 40 datasets de PMLB, que
no traen particiones propias, se generan 5 particiones, estratificadas en clasificación,
barajadas y con semilla fija. Se eligieron 5 porque era el valor especificado por defecto en la
configuración del estudio. Coincide con uno de los dos valores que recomienda ISLR (k = 5 o
k = 10) y, con datasets de 50 a 500 filas, deja al menos 10 observaciones en cada partición de
prueba.

Que unos datasets usen 10 particiones y otros 5 no sesga la comparación, porque todas las
métricas se calculan dentro de cada dataset antes de agregarse y nunca se compara la puntuación
de un dataset con la de otro. Dos métodos se consideran empatados cuando su puntuación media
está a menos de una desviación estándar (la del mejor método) de la del mejor, de modo que en un
dataset puede haber varios mejores métodos. Ese ranking real es la referencia contra la que se
mide cualquier estrategia de selección.

La métrica decide cuál es el mejor método en cada dataset, y todo el estudio se apoya en esa
respuesta, así que se eligió con dos condiciones: que no diera por bueno un modelo trivial y que
permitiera promediar resultados entre datasets distintos.

En clasificación se usa la *balanced accuracy*, la media del *recall* de cada clase. La colección
tiene datasets con clases muy desequilibradas, y en ellos la *accuracy* normal pondría en cabeza
a un modelo que predice siempre la clase mayoritaria. La *balanced accuracy* no tiene ese
problema, sirve igual en problemas binarios y multiclase, va de 0 a 1 y, cuando las clases están
equilibradas, coincide con la *accuracy*. Se descartó el ROC-AUC porque necesita probabilidades, y
obtenerlas de la SVM obliga a una validación cruzada interna que multiplica el coste del método
más caro. El F1-macro y el coeficiente de Matthews eran defendibles, pero más difíciles de
explicar sin aportar nada en este caso.

En regresión se usa el R². Para decidir qué método gana dentro de un dataset la escala da igual,
pero el *regret* promedia diferencias de rendimiento entre datasets. Con el RMSE o el MAE eso
supondría sumar errores en unidades distintas, por ejemplo precios de viviendas con
concentraciones químicas. El R² no depende de la escala y es además la métrica que usa ISLR. Se
acota en 0 para que un modelo muy malo, con un R² muy negativo, no arrastre las medias.


<a id="grafico-3"></a>

El Gráfico 3 cuenta en cuántos datasets cada método es el mejor o empata con él, distinguiendo
las victorias en solitario (azul oscuro) de los empates (azul claro). Explica la fuerza de la
estrategia fija: si un método queda casi siempre entre los mejores, recomendarlo siempre ya
acierta mucho.

In [7]:
ms0 = METODOS_SCORES[METODOS_SCORES.missing_rate == 0.0].copy()
ganadores_por_dataset = ms0.groupby("dataset").winner.sum()
ms0["tipo_victoria"] = np.where(
    ms0.dataset.map(ganadores_por_dataset) == 1, "Único mejor", "Empatado entre los mejores")
victorias = (ms0[ms0.winner].groupby(["method", "tipo_victoria"]).size()
             .unstack(fill_value=0).reindex(columns=["Único mejor", "Empatado entre los mejores"],
                                            fill_value=0))
aplicable = ms0.groupby("method").dataset.nunique()
victorias["total"] = victorias.sum(axis=1)
victorias = victorias.reindex(aplicable.index, fill_value=0).sort_values("total")
etiquetas = [ETIQUETA_METODO.get(m, m) for m in victorias.index]

fig = go.Figure()
for columna, color in (("Único mejor", AZULES[5]), ("Empatado entre los mejores", AZULES[2])):
    fig.add_bar(
        y=etiquetas, x=victorias[columna], name=columna, orientation="h",
        marker=dict(color=color, line=dict(width=2, color=FONDO)),
        customdata=np.stack([victorias.total, aplicable.reindex(victorias.index)], axis=1),
        hovertemplate="<b>%{y}</b><br>" + columna + ": %{x}<br>"
                      "Mejor (total): %{customdata[0]} de %{customdata[1]} datasets "
                      "en los que se ejecutó<extra></extra>",
    )
fig.update_layout(barmode="stack")
fig.update_xaxes(title="Datasets en los que es (uno de) los mejores", showgrid=True,
                 gridcolor=REJILLA)
fig.update_yaxes(showgrid=False)
estilo(fig, f"Gráfico 3 · {ETIQUETA_METODO[victorias.total.idxmax()]} "
            "es el método que más veces resulta mejor",
       "Datasets en los que cada método es el mejor o empata con él "
       "(regla de 1 desviación estándar)",
       alto=640).show()

mas_ganador = victorias.total.idxmax()
print(f"Método más frecuentemente mejor: {ETIQUETA_METODO[mas_ganador]} "
      f"({victorias.total.max()} de {aplicable[mas_ganador]} datasets en los que se ejecutó)")

Método más frecuentemente mejor: Random Forest (66 de 106 datasets en los que se ejecutó)


<a id="grafico-4"></a>

El Gráfico 4 es un mapa de calor con los métodos en filas y los datasets en columnas. Cada celda
muestra cuánto rendimiento pierde ese método frente al mejor de ese dataset: los tonos claros
indican que queda cerca del mejor, los oscuros que queda lejos, y las celdas en blanco que el
método no aplica o no pudo ejecutarse. Permite ver de un vistazo que ningún método gana siempre,
pero que unos pocos casi nunca quedan lejos.

In [8]:
# Pérdida de cada método frente al mejor de su dataset. 0 = es el mejor; en blanco = el método
# no aplica a la tarea o no pudo ejecutarse. Recortado en 0.25 para que la escala sea legible.
TOPE = 0.25
matriz = ms0.pivot(index="method", columns="dataset", values="regret").clip(upper=TOPE)
orden_metodos = matriz.mean(axis=1).sort_values().index
tarea_ds = METAFEATURES.set_index("dataset").task
orden_datasets = sorted(matriz.columns, key=lambda d: (tarea_ds[d] != "regression",
                                                       ganadores_por_dataset[d]))
matriz = matriz.loc[orden_metodos, orden_datasets]

fig = go.Figure(go.Heatmap(
    z=matriz.values, x=matriz.columns, y=[ETIQUETA_METODO.get(m, m) for m in matriz.index],
    colorscale=[[i / (len(AZULES) - 1), c] for i, c in enumerate(AZULES)],
    zmin=0, zmax=TOPE, xgap=1, ygap=1,
    colorbar=dict(title=dict(text="Pérdida frente<br>al mejor", side="top"), thickness=12,
                  tickvals=[0, 0.1, 0.2, TOPE], ticktext=["0 (mejor)", "0.10", "0.20", "≥ 0.25"]),
    hovertemplate="<b>%{y}</b> en %{x}<br>Pérdida frente al mejor: %{z:.3f}<extra></extra>",
))
n_reg = int((tarea_ds[orden_datasets] == "regression").sum())
fig.add_vline(x=n_reg - 0.5, line=dict(color=TINTA_2, width=1.5, dash="dot"))
fig.add_annotation(x=n_reg / 2, y=1.0, yref="paper", yanchor="bottom", text="<b>Regresión</b>",
                   showarrow=False)
fig.add_annotation(x=n_reg + (len(orden_datasets) - n_reg) / 2, y=1.0, yref="paper",
                   yanchor="bottom", text="<b>Clasificación</b>", showarrow=False)
fig.update_xaxes(showticklabels=False,
                 title="Datasets (ordenados por tarea y nº de métodos empatados)")
fig.update_yaxes(showgrid=False, autorange="reversed")
estilo(fig, "Gráfico 4 · Ningún método gana siempre, pero unos pocos casi nunca quedan lejos",
       "Pérdida de cada método frente al mejor de cada dataset · "
       "métodos ordenados de menor a mayor pérdida media",
       alto=680).show()

<a id="metricas"></a>

### Cómo se mide cada estrategia

Las estrategias, definidas en la sección 0, se evalúan sobre ese ranking real con tres métricas.
El *hit rate* es la proporción de datasets en los que la primera recomendación está entre los
mejores métodos. El *top-3* es la proporción en la que algún mejor método aparece entre las tres
primeras sugerencias, que es lo que el usuario ve en la aplicación. El *regret* es el
rendimiento que se pierde por seguir la primera recomendación en lugar de usar el mejor método.
Un método que no puede ejecutarse pierde la puntuación entera. Está en la misma escala que la
métrica, de 0 a 1, así que un regret de 0,03 significa perder tres centésimas de *balanced
accuracy* o de R².

Los resultados se presentan en dos grupos: todos los datasets (106) y solo aquellos en los que
la elección importa (51). Dos ejemplos de la colección ayudan a ver la diferencia. En
`forest_fires`, que consiste en predecir el área quemada en incendios forestales, ningún método
consigue predecir nada: el mejor obtiene un R² de 0,019 y los peores 0,000, así que la regla de
empate cuenta a los 16 métodos como mejores y cualquier estrategia acierta, incluso eligiendo a
ciegas. En `611_fri_c3_100_5`, en cambio, la red neuronal obtiene 0,892 y el segundo mejor
método, KNN, 0,758, de modo que solo acierta quien elige la red neuronal. Si se promedian los
dos tipos de dataset, los del primero regalan un acierto a todas las estrategias por igual y las
diferencias entre ellas se diluyen, igual que en un examen en el que la mitad de las preguntas
diera por buena cualquier respuesta. La elección al azar lo deja claro: acierta en el 49 % de
los datasets del primer tipo y solo en el 2 % de los del segundo. De media, empatan con el mejor
5,2 de los 14,7 métodos que se ejecutan en cada dataset, y en 4 datasets empatan todos.

Los 51 datasets son aquellos en los que empatan como mucho tres métodos. El umbral no se eligió
para mejorar los resultados: es el número de sugerencias que muestra la aplicación, y si empatan
más de tres métodos, un usuario que siga cualquiera de ellas acierta seguro. La separación
afecta solo a la evaluación, porque todas las estrategias se entrenan siempre con todos los
datasets, y los dos grupos se informan siempre juntos porque responden a preguntas distintas:
los 106 dicen con qué frecuencia importa elegir bien, y los 51 si la estrategia acierta cuando
importa.

Esta separación no estaba en el planteamiento inicial. Surgió al revisar la primera evaluación,
hecha con 44 datasets, en la que la elección al azar acertaba en las tres primeras sugerencias
el 75 % de las veces, algo imposible si de verdad hubiera que acertar un único método entre
quince. La causa eran los empates, y a partir de ahí se decidió informar siempre de los dos
grupos. Aunque la decisión se tomó después de ver esos primeros resultados, no los favoreció: el
umbral quedó fijado por la interfaz y no se volvió a tocar, y la ventaja que la separación
parecía dar entonces a la estrategia aprendida desapareció al ampliar la colección a 106
datasets. El razonamiento completo está en la decisión D-036 de
[`DECISIONS.md`](https://github.com/cecimerelo/ml-sandbox/blob/main/DECISIONS.md).

<a id="resultados"></a>

## 3. Resultados y contraste de las hipótesis

<a id="tabla-resultados"></a>

La tabla de resultados principal recoge el hit rate, el top-3 y el regret medios de cada
estrategia en los dos estratos. Es la tabla que responde a la hipótesis, y los gráficos
siguientes la desglosan.

In [9]:
def resumen(frame):
    return frame.groupby("estrategia").agg(
        datasets=("dataset", "nunique"), hit_rate=("hit", "mean"),
        top3=("top_k_hit", "mean"), regret=("regret", "mean")).reindex(ORDEN_ESTRATEGIAS)

tabla = pd.concat({"Todos los datasets": resumen(OUT0),
                   "Donde la elección importa": resumen(OUT0[OUT0.discriminating])})
tabla.round(3)

datasets  hit_rate   top3  \
                          estrategia                                       
Todos los datasets        Aprendida (Layer 2)       106     0.632  0.783   
                          Método único fijo         106     0.623  0.745   
                          Heurísticas (ISLR)        106     0.321  0.679   
                          Aleatoria                 106     0.264  0.594   
Donde la elección importa Aprendida (Layer 2)        51     0.490  0.686   
                          Método único fijo          51     0.471  0.647   
                          Heurísticas (ISLR)         51     0.176  0.490   
                          Aleatoria                  51     0.020  0.294   

                                               regret  
                          estrategia                   
Todos los datasets        Aprendida (Layer 2)   0.041  
                          Método único fijo     0.036  
                          Heurísticas (ISLR)    0.116  
                          Aleatoria             0.244  
Donde la elección importa Aprendida (Layer 2)   0.038  
                          Método único fijo     0.031  
                          Heurísticas (ISLR)    0.112  
                          Aleatoria             0.325

<a id="grafico-5"></a>

El Gráfico 5 muestra los mismos aciertos de la tabla con su intervalo de confianza al 95 %,
calculado por *bootstrap*: se remuestrean los datasets 2.000 veces y se mide cuánto varía el
resultado. Cuando los intervalos de dos estrategias se solapan mucho, la diferencia entre ellas
puede deberse al azar de qué datasets entraron en la muestra.

In [10]:
RNG = np.random.default_rng(20260815)
REPETICIONES = 2000

def intervalo(valores, rng=RNG, reps=REPETICIONES):
    # IC 95 % bootstrap sobre datasets: se remuestrean datasets, no evaluaciones.
    valores = np.asarray(valores, dtype=float)
    medias = rng.choice(valores, size=(reps, len(valores)), replace=True).mean(axis=1)
    return np.percentile(medias, [2.5, 97.5])

filas = []
for estrato, frame in (("Todos los datasets (106)", OUT0),
                       ("Donde la elección importa (51)", OUT0[OUT0.discriminating])):
    for metrica, columna in (("Hit rate", "hit"), ("Top-3", "top_k_hit")):
        for estrategia in ORDEN_ESTRATEGIAS:
            valores = frame.loc[frame.estrategia == estrategia, columna]
            bajo, alto = intervalo(valores)
            filas.append(dict(estrato=estrato, metrica=metrica, estrategia=estrategia,
                              valor=valores.mean(), bajo=bajo, alto=alto))
ic = pd.DataFrame(filas)

estratos = ic.estrato.unique().tolist()
metricas = ["Hit rate", "Top-3"]
fig = make_subplots(rows=2, cols=2, shared_yaxes=True, vertical_spacing=0.16,
                    horizontal_spacing=0.05,
                    subplot_titles=[f"{m} · {e}" for m in metricas for e in estratos])
for i, metrica in enumerate(metricas, start=1):
    for j, estrato in enumerate(estratos, start=1):
        sub = ic[(ic.metrica == metrica) & (ic.estrato == estrato)]
        for _, fila in sub.iterrows():
            fig.add_bar(
                x=[fila.estrategia], y=[fila.valor], name=fila.estrategia,
                marker=dict(color=COLOR_ESTRATEGIA[fila.estrategia]),
                error_y=dict(type="data", symmetric=False, array=[fila.alto - fila.valor],
                             arrayminus=[fila.valor - fila.bajo], color=TINTA_2, thickness=1.5),
                text=[f"{fila.valor:.2f}"], textposition="inside", insidetextanchor="start",
                textfont=dict(color="white"),
                showlegend=(i == 1 and j == 1), legendgroup=fila.estrategia,
                hovertemplate=f"<b>{fila.estrategia}</b><br>{metrica}: {fila.valor:.2f}<br>"
                              f"IC 95 %: [{fila.bajo:.2f}, {fila.alto:.2f}]<extra></extra>",
                row=i, col=j)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(range=[0, 1.08], tickformat=".0%")
estilo(fig, "Gráfico 5 · La recomendación personalizada empata con la elección fija",
       "Proporción de aciertos por estrategia, con intervalo de confianza bootstrap al 95 %",
       alto=720).show()

<a id="contraste-principal"></a>

Para contrastar la hipótesis principal se comparan la estrategia aprendida y el método único
fijo sobre los mismos datasets, como datos pareados. El hit rate y el top-3 se contrastan con el
test exacto de McNemar, que solo tiene en cuenta los datasets en los que una estrategia acierta
y la otra no, y el regret con el test de rangos con signo de Wilcoxon sobre la diferencia de
pérdida en cada dataset. Se considera significativa una diferencia cuando p es menor que 0,05.

In [11]:
def pareado(frame, a="Aprendida (Layer 2)", b="Método único fijo"):
    ancho = frame.pivot(index="dataset", columns="estrategia",
                        values=["hit", "top_k_hit", "regret"])
    filas = []
    for metrica, columna in (("Hit rate", "hit"), ("Top-3", "top_k_hit")):
        acierta = ancho[columna].astype(bool)
        gana_a = int((acierta[a] & ~acierta[b]).sum())
        gana_b = int((~acierta[a] & acierta[b]).sum())
        p = stats.binomtest(gana_a, gana_a + gana_b).pvalue if gana_a + gana_b else 1.0
        filas.append(dict(métrica=metrica, **{f"gana {a}": gana_a, f"gana {b}": gana_b},
                          p_valor=p))
    diferencia = (ancho["regret"][a] - ancho["regret"][b]).astype(float)
    distintos = diferencia[diferencia != 0]
    p = stats.wilcoxon(distintos).pvalue if len(distintos) else 1.0
    filas.append(dict(métrica="Regret", **{f"gana {a}": int((diferencia < 0).sum()),
                                            f"gana {b}": int((diferencia > 0).sum())},
                      p_valor=p))
    return pd.DataFrame(filas).set_index("métrica")

contraste = pd.concat({"Todos los datasets": pareado(OUT0),
                       "Donde la elección importa": pareado(OUT0[OUT0.discriminating])})
contraste.style.format({"p_valor": "{:.3f}"})

<a id="contraste-secundaria"></a>

La hipótesis secundaria se contrasta con los mismos tests pareados, comparando las heurísticas
de ISLR con la elección aleatoria y, como referencia, con el método único fijo.

In [12]:
def frente_a(frame, rival, a="Heurísticas (ISLR)"):
    tabla = pareado(frame, a=a, b=rival)
    return tabla.set_axis(["gana heurísticas", "gana rival", "p_valor"], axis=1)

contraste_heuristicas = pd.concat({
    (estrato, rival): frente_a(frame, rival)
    for estrato, frame in (("Todos los datasets", OUT0),
                           ("Donde la elección importa", OUT0[OUT0.discriminating]))
    for rival in ("Aleatoria", "Método único fijo")})
contraste_heuristicas.index.names = ["estrato", "rival", "métrica"]
contraste_heuristicas.style.format({"p_valor": "{:.3f}"})

<a id="grafico-6"></a>

La media del regret no distingue entre fallar un poco muchas veces y fallar mucho unas pocas.
En el Gráfico 6 cada punto es un dataset y la caja resume su distribución, de modo que se ve si
una estrategia comete errores graves, que aparecen como puntos altos, o solo pequeños. Las
heurísticas y el azar llegan a perder casi toda la puntuación en algunos datasets. También se ve
que el peor fallo de la estrategia aprendida es mayor que el de la fija, 0,50 frente a 0,28
sobre los 106 datasets y 0,39 frente a 0,19 sobre los 51, lo que
explica que su regret medio sea algo más alto pese a acertar con la misma frecuencia.

In [13]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=["Todos los datasets (106)", "Donde la elección importa (51)"])
for j, frame in enumerate((OUT0, OUT0[OUT0.discriminating]), start=1):
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = frame[frame.estrategia == estrategia]
        fig.add_box(
            x=[estrategia] * len(sub), y=sub.regret, name=estrategia,
            marker=dict(color=COLOR_ESTRATEGIA[estrategia], size=7, opacity=0.6,
                        line=dict(width=1, color=FONDO)),
            line=dict(color=COLOR_ESTRATEGIA[estrategia], width=2),
            fillcolor="rgba(0,0,0,0)", boxpoints="all", jitter=0.45, pointpos=0,
            boxmean=True, customdata=sub.dataset,
            hovertemplate="<b>%{customdata}</b><br>Regret: %{y:.3f}"
                          "<extra>" + estrategia + "</extra>",
            showlegend=(j == 1), legendgroup=estrategia, row=1, col=j)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(title_text="Regret (rendimiento perdido)", row=1, col=1)
estilo(fig, "Gráfico 6 · Las heurísticas y el azar fallan a veces por mucho",
       "Regret por dataset · caja = cuartiles, línea discontinua = media, cada punto un dataset",
       alto=560).show()

<a id="grafico-7"></a>

El Gráfico 7 compara, dataset a dataset, lo que pierden la estrategia aprendida y la fija. Los
puntos sobre la diagonal son empates, los de debajo son datasets en los que gana la aprendida y
los de encima, datasets en los que gana la fija. Un empate en la media podría esconder que cada
una gana en datasets distintos, y este gráfico permite verlo.

In [14]:
ancho = OUT0.pivot(index="dataset", columns="estrategia", values="regret")
ancho = ancho.join(OUT0.drop_duplicates("dataset").set_index("dataset")[["discriminating"]])
a, b = ancho["Aprendida (Layer 2)"], ancho["Método único fijo"]
ancho["resultado"] = np.select([np.isclose(a, b), a < b, a > b],
                               ["Igual regret", "Aprendida pierde menos", "Fija pierde menos"],
                               default="Igual regret")
conteo = ancho.resultado.value_counts()
colores = {"Aprendida pierde menos": AZUL, "Fija pierde menos": NARANJA,
           "Igual regret": TINTA_3}
tope = float(max(a.max(), b.max())) * 1.05

fig = go.Figure()
fig.add_scatter(x=[0, tope], y=[0, tope], mode="lines", showlegend=False, hoverinfo="skip",
                line=dict(color=EJE, width=1.5, dash="dash"))
for resultado, color in colores.items():
    sub = ancho[ancho.resultado == resultado]
    fig.add_scatter(
        x=sub["Método único fijo"], y=sub["Aprendida (Layer 2)"], mode="markers",
        name=f"{resultado} ({conteo.get(resultado, 0)})",
        marker=dict(color=color, size=10, opacity=0.8, line=dict(width=1.5, color=FONDO)),
        customdata=sub.index,
        hovertemplate="<b>%{customdata}</b><br>Fija: %{x:.3f}<br>"
                      "Aprendida: %{y:.3f}<extra></extra>")
fig.add_annotation(x=tope * 0.75, y=tope * 0.2, text="La aprendida pierde menos ↓",
                   showarrow=False, font=dict(color=TINTA_2))
fig.add_annotation(x=tope * 0.2, y=tope * 0.8, text="↑ La fija pierde menos",
                   showarrow=False, font=dict(color=TINTA_2))
fig.update_xaxes(title="Regret del método único fijo", range=[-0.01, tope], showgrid=True,
                 gridcolor=REJILLA, constrain="domain")
fig.update_yaxes(title="Regret de la estrategia aprendida", range=[-0.01, tope],
                 scaleanchor="x", scaleratio=1, constrain="domain")
estilo(fig, "Gráfico 7 · Dataset a dataset, ninguna de las dos domina",
       f"Cada punto es un dataset ({len(ancho)}). Sobre la diagonal, ambas pierden lo mismo",
       alto=680).show()

<a id="grafico-8"></a>

El Gráfico 8 muestra el hit rate de cada estrategia según el número de filas del dataset, para
comprobar si la personalización ayuda en algún régimen concreto aunque no lo haga en conjunto.
En las dos bandas de menos de 10.000 filas ambas aciertan exactamente lo mismo. En la de más de
10.000 la aprendida queda ligeramente por encima, pero por un único dataset, una
diferencia que no es significativa. La tabla final contrasta la banda de 500 a 10.000 filas,
donde una versión anterior del estudio parecía mostrar una ventaja.

In [15]:
por_banda = (OUT0.groupby(["rows", "estrategia"])
             .agg(hit_rate=("hit", "mean"), datasets=("dataset", "nunique")).reset_index())
por_banda_d = (OUT0[OUT0.discriminating].groupby(["rows", "estrategia"])
               .agg(hit_rate=("hit", "mean"), datasets=("dataset", "nunique")).reset_index())

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=["Todos los datasets", "Donde la elección importa"])
for j, frame in enumerate((por_banda, por_banda_d), start=1):
    n = frame.groupby("rows").datasets.first().reindex(BANDAS_FILAS)
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = frame[frame.estrategia == estrategia].set_index("rows").reindex(BANDAS_FILAS)
        fig.add_bar(
            x=[f"{banda}<br><span style='font-size:11px'>n = {n[banda]}</span>"
               for banda in BANDAS_FILAS],
            y=sub.hit_rate, name=estrategia, marker=dict(color=COLOR_ESTRATEGIA[estrategia]),
            marker_line=dict(width=2, color=FONDO), showlegend=(j == 1), legendgroup=estrategia,
            hovertemplate="<b>" + estrategia
                          + "</b><br>Filas %{x}<br>Hit rate: %{y:.2f}<extra></extra>",
            row=1, col=j)
fig.update_layout(barmode="group", bargap=0.25)
fig.update_xaxes(title_text="Filas del dataset")
fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="Hit rate", row=1, col=1)
estilo(fig, "Gráfico 8 · Por tamaño de muestra, la aprendida y la fija "
            "siguen sin diferencias significativas",
       "Hit rate por banda de filas del dataset", alto=520).show()

(OUT0[OUT0.discriminating & (OUT0.rows == "500-10k")]
   .pipe(pareado).style.format({"p_valor": "{:.3f}"})
   .set_caption("Contraste pareado en la banda 500–10k, donde la elección importa"))

,gana Aprendida (Layer 2),gana Método único fijo,p_valor
métrica,,,
Hit rate,2,2,1.000
Top-3,2,0,0.500
Regret,5,3,0.641


<a id="robustez"></a>

## 4. Robustez

### 4.1 Valores faltantes

El benchmark se repitió eliminando al azar un 5 % y un 25 % de las celdas de los predictores,
para comprobar si la conclusión se mantiene cuando los datos están incompletos. Los huecos se
inyectan de forma completamente aleatoria (MCAR), el supuesto más neutro: en datos reales la
ausencia suele seguir un patrón, así que este experimento es un punto de partida y no cubre esos
casos más difíciles. Hay además una asimetría entre estrategias: la aprendida se entrena siempre
con los resultados sin valores faltantes, mientras que la fija calcula su orden con los tres
niveles mezclados. No cambia la conclusión, pero conviene tenerla en cuenta al leer estos
resultados.

<a id="grafico-9"></a>

El Gráfico 9 muestra el hit rate y el regret de cada estrategia con un 0 %, un 5 % y un 25 % de
celdas faltantes, sobre los 106 datasets, y la tabla que lo acompaña recoge los valores exactos.

In [16]:
por_tasa = (OUT.groupby(["missing_rate", "estrategia"])
            .agg(hit_rate=("hit", "mean"), regret=("regret", "mean")).reset_index())

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["Hit rate (más es mejor)", "Regret (menos es mejor)"])
for j, metrica in enumerate(("hit_rate", "regret"), start=1):
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = por_tasa[por_tasa.estrategia == estrategia]
        fig.add_scatter(
            x=sub.missing_rate, y=sub[metrica], name=estrategia, mode="lines+markers",
            line=dict(color=COLOR_ESTRATEGIA[estrategia], width=2),
            marker=dict(size=9, line=dict(width=1.5, color=FONDO)),
            showlegend=(j == 1), legendgroup=estrategia,
            hovertemplate="<b>" + estrategia + "</b><br>Faltantes: %{x:.0%}<br>"
                          + ("Hit rate: %{y:.2f}" if metrica == "hit_rate" else "Regret: %{y:.3f}")
                          + "<extra></extra>",
            row=1, col=j)
fig.update_xaxes(tickvals=[0, 0.05, 0.25], tickformat=".0%", title_text="Celdas faltantes")
fig.update_yaxes(tickformat=".0%", range=[0, 1], row=1, col=1)
fig.update_yaxes(rangemode="tozero", row=1, col=2)
estilo(fig, "Gráfico 9 · El empate se mantiene con datos incompletos",
       "Todos los datasets (106) con 0 %, 5 % y 25 % de celdas faltantes", alto=500).show()

(por_tasa.pivot(index="estrategia", columns="missing_rate", values=["hit_rate", "regret"])
   .reindex(ORDEN_ESTRATEGIAS).round(3))

hit_rate               regret              
missing_rate            0.00   0.05   0.25   0.00   0.05   0.25
estrategia                                                     
Aprendida (Layer 2)    0.632  0.726  0.689  0.041  0.038  0.035
Método único fijo      0.623  0.764  0.689  0.036  0.032  0.033
Heurísticas (ISLR)     0.321  0.302  0.292  0.116  0.125  0.132
Aleatoria              0.264  0.302  0.274  0.244  0.234  0.208

La celda siguiente repite el test pareado de la hipótesis principal, la estrategia aprendida
frente a la fija, para cada porcentaje de celdas faltantes.

In [17]:
contrastes_tasa = pd.concat({
    f"{tasa:.0%} faltantes": pareado(OUT[OUT.missing_rate == tasa])
    for tasa in sorted(OUT.missing_rate.unique())})
contrastes_tasa.style.format({"p_valor": "{:.3f}"}).set_caption(
    "Aprendida frente a fija, por nivel de faltantes (todos los datasets)")

<a id="interpretabilidad"></a>

### 4.2 Coste de exigir interpretabilidad

El recomendador permite al usuario pedir un modelo explicable. Si indica que la interpretabilidad
es algo importante, se descartan los métodos opacos (bagging, random forest, boosting, SVM RBF y
redes neuronales). Si la considera crítica, se descartan también los que solo se explican con
esfuerzo (KNN, QDA, PCR, PLS, regresión polinómica con y sin interacciones y splines). Este
apartado mide cuánto rendimiento cuesta esa restricción.

El Gráfico 10 separa dos cantidades para cada nivel de exigencia. La barra gris es el coste
inevitable de la restricción: cuánto rendimiento se pierde, de media, al usar el mejor método
permitido en lugar del mejor sin restricciones. La barra azul es lo que pierde realmente el
recomendador cuando se le impone esa restricción. La diferencia entre ambas no se debe a la
interpretabilidad, sino a que el recomendador no elige el mejor de los métodos permitidos.

In [18]:
# Qué métodos descarta cada nivel de interpretabilidad (la misma regla que usa el recomendador).
EXCLUYE = {"Sin restricción": set(), "Algo importante": {"opaque"},
           "Crítica": {"opaque", "with effort"}}
CLASE = dict(zip(METODOS.name, METODOS.explainability, strict=True))

# Coste inevitable: cuánto queda el mejor método permitido por debajo del mejor sin restricción.
mejor = ms0.groupby("dataset").score.max()
inevitable = {
    nivel: (mejor - ms0[~ms0.method.map(CLASE).isin(excluidas)]
            .groupby("dataset").score.max().reindex(mejor.index)).mean()
    for nivel, excluidas in EXCLUYE.items()}

niveles = {n["explainability"]: n for n in EXPLICABILIDAD["levels"]}
recomendador = {"Sin restricción": niveles["somewhat"]["unconstrained_mean_regret"],
                "Algo importante": niveles["somewhat"]["mean_regret"],
                "Crítica": niveles["critical"]["mean_regret"]}
total = RESULTADOS.dataset.nunique()

fig = go.Figure()
for nombre, valores, color in (("Mejor método permitido (coste inevitable)", inevitable, TINTA_3),
                               ("Método que elige el recomendador", recomendador, AZUL)):
    fig.add_bar(
        x=list(valores), y=list(valores.values()), name=nombre,
        marker=dict(color=color, line=dict(width=2, color=FONDO)),
        text=[f"{v:.3f}" for v in valores.values()], textposition="outside", cliponaxis=False,
        textfont=dict(color=TINTA),
        hovertemplate="<b>%{x}</b><br>" + nombre + "<br>Regret medio: %{y:.3f}<extra></extra>")
fig.update_layout(barmode="group", bargap=0.3)
fig.update_xaxes(title="Importancia de la interpretabilidad")
fig.update_yaxes(title="Regret medio", rangemode="tozero")
estilo(fig, "Gráfico 10 · La restricción cuesta poco; "
            "el recomendador elige mal entre los métodos interpretables",
       f"Regret medio sobre {total} datasets · gris: lo mínimo que se pierde · "
       "azul: lo que pierde el recomendador",
       alto=500).show()

pd.DataFrame({"Coste inevitable": inevitable, "Recomendador": recomendador}).round(3)

,Coste inevitable,Recomendador
Sin restricción,0.000,0.041
Algo importante,0.027,0.156
Crítica,0.061,0.146


<a id="conclusiones"></a>

## 5. Conclusiones

La hipótesis principal no se verifica. Personalizar la recomendación según las características
del problema no acierta más que recomendar siempre el mismo método (Gráficos 5 y 7): los hit
rates son prácticamente iguales, las diferencias no son significativas en ninguna métrica y el
resultado se mantiene por tamaño de muestra (Gráfico 8) y con datos incompletos (Gráfico 9).

La hipótesis secundaria se verifica solo en parte. Frente al azar, las heurísticas de ISLR
pierden significativamente menos rendimiento en los dos estratos, pero solo aciertan el mejor
método con más frecuencia en los datasets donde la elección importa. Sobre el conjunto de los
106, la diferencia en aciertos no es significativa, porque allí el azar acierta a menudo gracias
a los empates. Frente a la elección fija, en cambio, quedan claramente por detrás, tanto en
aciertos como en rendimiento perdido (Gráficos 5 y 6 y contraste de la sección 3).

Hay además un hallazgo que no se había anticipado. El propio benchmark revela cuál es esa
elección fija tan difícil de batir, Random Forest, con Gradient Boosting muy cerca (Gráfico 3),
y con ello muestra que en datos tabulares típicos elegir bien el método importa menos de lo que
se suele suponer (Gráfico 4).

Por último, exigir un modelo explicable cambia la elección en casi todos los datasets, porque el
mejor método disponible suele ser opaco, pero su coste inevitable es pequeño: el mejor método
interpretable queda de media a 0,03 del mejor si se descartan los opacos y a 0,06 si se descartan
también los que solo se explican con esfuerzo. Lo que sí es grande es la pérdida del recomendador
con esas restricciones, entre dos y seis veces mayor que la inevitable, porque elige mal entre los
métodos interpretables: aprendió a ordenar bien los métodos potentes, pero no los sencillos
(Gráfico 10). Es la principal mejora pendiente del recomendador.

<a id="anexo"></a>

## Anexo 1 — Recalcular todo desde los resultados

Los ficheros `outcomes.parquet`, `method_scores.parquet` y `explainability.json` se generan a
partir del parquet de resultados con el código del repositorio. Para recalcularlos, lo que lleva
entre 15 y 20 minutos y requiere Python 3.12, basta con cambiar `RECALCULAR` a `True` y volver a
ejecutar el cuaderno desde el principio después de esta celda.

In [19]:
RECALCULAR = False

if RECALCULAR:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
    subprocess.run([sys.executable, "scripts/export_outcomes.py"], cwd=ROOT, check=True)

<a id="anexo-2"></a>

## Anexo 2 — Decisiones metodológicas y trabajo futuro

### Decisiones metodológicas

Cada decisión del estudio está registrada, con sus alternativas y su razonamiento, en
[`DECISIONS.md`](https://github.com/cecimerelo/ml-sandbox/blob/main/DECISIONS.md). Este apartado
resume las que más influyen en los resultados.

La colección combina OpenML-CC18 y OpenML-CTR23, colecciones de referencia en la literatura, con
PMLB para cubrir los datasets de menos de 500 filas que OpenML apenas ofrece (D-013 y D-025). Se
limitó a datasets de entre 50 y 100.000 filas, porque por debajo de 50 la validación cruzada
deja particiones demasiado pequeñas y por encima un solo dataset consumiría más cómputo que toda
la banda pequeña (D-016), y cada dataset se evalúa como mucho sobre 20.000 filas, muestreadas
con la semilla del estudio, lo que no altera la comparación porque el límite se aplica igual a
todos los métodos (D-030). No se publica una copia de los datos, porque la licencia de la
mayoría de los datasets de PMLB no puede verificarse. Se publica en cambio la receta exacta para
reconstruir la colección (D-017).

Los métodos se evalúan con *balanced accuracy* en clasificación, que no premia predecir siempre
la clase mayoritaria en los datasets desequilibrados, y con R² acotado en 0 en regresión
(D-021). Solo se ajustan por validación cruzada interna los hiperparámetros que un método
necesita para estar definido, como la penalización de Ridge o Lasso. El resto usa los valores
por defecto de la librería, que es lo que hace ISLR (D-019). Las particiones de OpenML son las
suyas predefinidas, para poder comparar con la literatura (D-003). Los valores faltantes se
inyectan de forma controlada porque PMLB no los tiene (D-015), y cada método los trata como lo
haría en la práctica: los que los admiten reciben los datos tal cual y los demás imputan dentro
de su propio *pipeline*, ajustado en cada partición de entrenamiento (D-022).

El recomendador aprende solo de propiedades que el usuario puede indicar en el formulario, para
que funcione también sin subir datos (D-026 y D-028), y las recibe agrupadas en las mismas
bandas que ofrece el formulario, para que el modelo vea lo mismo al entrenarse que al usarse
(D-027). Predice la pérdida relativa de cada método y no el ganador (D-033). Se compara con dos
referencias, la elección aleatoria y el método único fijo, que es la que decide si personalizar
merece la pena (D-031). Como cota superior estaba previsto AMLBID, un sistema de recomendación
de métodos publicado en 2022 (Garouani et al.), pero se descartó porque no pudo reproducirse:
falla con los datasets de la colección (D-029). Los resultados se presentan en dos estratos
porque, sobre todos los datasets, los empates hacen que casi cualquier estrategia acierte y la
media oculta las diferencias (D-036). Por último, la interpretabilidad se trata como una
propiedad de cada método, con tres niveles, y actúa como filtro que el usuario impone, no como
un criterio que el recomendador pueda compensar con precisión (D-035).

### Trabajo futuro

La línea más directa es mejorar el propio recomendador. El random forest de Layer 2 no se
comparó con otros modelos posibles, como *gradient boosting* o una regresión lineal con
interacciones, y alguno podría rendir mejor. Sobre todo, el Gráfico 10 muestra que el
recomendador elige mal entre los métodos interpretables: pierde entre dos y seis veces más de lo
que exige la restricción. Una opción sería entrenarlo y evaluarlo también dentro de cada
subconjunto de métodos permitidos, para que aprenda a ordenar los métodos sencillos y no solo
los potentes.

La segunda línea es ampliar la colección, sobre todo en los regímenes con pocos datasets: más de
50 predictores, alta dimensionalidad, más de 10.000 filas y valores faltantes de origen. Con más
datasets podría comprobarse además si la ventaja de la estrategia aprendida en top-3, que se ha
mantenido en todas las versiones del estudio sin llegar a ser significativa, es real.

Queda también por estudiar el efecto de los valores faltantes con patrones más realistas que el
azar completo (MCAR), en los que la ausencia depende de los propios datos, y entrenando el
recomendador con los resultados de cada nivel de faltantes en lugar de solo con los datos
completos.

Por último, el estudio podría completarse de tres formas. La primera es incorporar GAM y BART,
previstos a través de R (D-020) pero que no llegaron a ejecutarse en el benchmark. La segunda,
añadir una cota superior, como otro sistema de meta-aprendizaje, que indique cuánto margen de
mejora queda. La tercera, usar, cuando el usuario sube sus datos, meta-características
calculadas directamente sobre ellos en lugar de solo las que admite el formulario.